Download GT tracking data

In [10]:
!mkdir -p TrackEval/data/gt
file_id = "1Da3ew0XaWjW86o6NC_YpeQp9TA2OINJw"
output_name = "sportsmot.zip"
!cd TrackEval/data/gt && gdown $file_id -O $output_name
!unzip TrackEval/data/gt/sportsmot.zip -d TrackEval/data/gt
!rm TrackEval/data/gt/sportsmot.zip

Downloading...
From: https://drive.google.com/uc?id=1Da3ew0XaWjW86o6NC_YpeQp9TA2OINJw
To: /Users/alexanderbodner/Documents/roboflow/trackers metrics/sportsmot/TrackEval/data/gt/sportsmot.zip
100%|██████████████████████████████████████| 4.52M/4.52M [00:00<00:00, 14.2MB/s]
Archive:  TrackEval/data/gt/sportsmot.zip
replace TrackEval/data/gt/sportsmot/SportsMOT-train.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

OSError: [Errno 5] Input/output error

In [ ]:
import os
import itertools
import multiprocessing as mp

from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

import numpy as np
import pandas as pd
import supervision as sv

from trackers import OCSORTTracker
from trackers.eval import evaluate_mot_sequences


# Root directory for data and GT.\
# and `sportsmot_yolox_dets` is in the project root.
DATA_ROOT = "./"

SPORTSMOT_DET_ROOT = os.path.join(DATA_ROOT, "sportsmot_yolox_dets")
SPORTSMOT_GT_ROOT = os.path.join(DATA_ROOT, "TrackEval", "data", "gt", "sportsmot")

MIN_BOX_AREA = 10
VERTICAL_RATIO_THRESH = 1.6


def get_yolox_detections(frame_id, det_list):
    dets_per_frame = [d for d in det_list if d.split(",")[0] == str(frame_id)]

    dets = []
    for line in dets_per_frame:
        det = line.split(",")  # "frame_no,x1,y1,x2,y2,det_conf"
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])

    return dets


def build_dets_index(det_list):
    dets_by_frame = defaultdict(list)
    for line in det_list:
        frame_id = int(line.split(",")[0])
        dets_by_frame[frame_id].append(line)
    return dets_by_frame


def get_detections_from_dict(frame_id, dets_by_frame):
    dets = []
    for line in dets_by_frame[frame_id]:
        det = line.split(",")
        x1 = float(det[1])
        y1 = float(det[2])
        x2 = float(det[3])
        y2 = float(det[4])
        conf = float(det[5])
        dets.append([x1, y1, x2, y2, conf])
    return dets

In [2]:
def run_ocsort_sportsmot_once(
    split: str,
    lost_track_buffer: int,
    minimum_iou_threshold: float,
    minimum_consecutive_frames: int,
    direction_consistency_weight: float,
    high_conf_det_threshold: float,
    delta_t: int,
):
    """Run OC-SORT on one SportsMOT split (val or test) for a given hyperparameter set.

    Generates MOT-format result files and, if GT is available under
    `SPORTSMOT_GT_ROOT`, evaluates CLEAR, HOTA and Identity metrics via TrackEval.
    """

    assert split in {"val", "test"}, f"Unsupported split: {split}"

    tracker = OCSORTTracker(
        high_conf_det_threshold=high_conf_det_threshold,
        minimum_iou_threshold=minimum_iou_threshold,
        minimum_consecutive_frames=minimum_consecutive_frames,
        delta_t=delta_t,
        direction_consistency_weight=direction_consistency_weight,
        lost_track_buffer=lost_track_buffer,
    )

    det_root = os.path.join(SPORTSMOT_DET_ROOT, split)

    # Group all SportsMOT OC-SORT outputs under a single root directory
    outputs_root = "OCSORT_outputs_sportsmot"
    os.makedirs(outputs_root, exist_ok=True)

    save_dir = os.path.join(
        outputs_root,
        f"{split}_L{lost_track_buffer}_I{minimum_iou_threshold}__C{minimum_consecutive_frames}_"
        f"D{direction_consistency_weight}_H{high_conf_det_threshold}_T{delta_t}",
    )
    os.makedirs(save_dir, exist_ok=True)

    for seq in sorted(os.listdir(det_root)):
        if not seq.endswith(".txt"):
            continue

        tracker.reset()
        seq_name = seq.split(".")[0]

        with open(os.path.join(det_root, seq), "r") as f_det:
            det_list = f_det.readlines()
            dets_by_frame = build_dets_index(det_list)

        last_frame = int(det_list[-1].split(",")[0])
        output_lines = []

        for frame_id in range(1, last_frame + 1):
            raw_dets = get_detections_from_dict(frame_id, dets_by_frame)
            if raw_dets:
                raw_dets = np.array(raw_dets)
                dets = sv.Detections(
                    xyxy=raw_dets[:, :4],
                    confidence=raw_dets[:, 4],
                )
            else:
                dets = sv.Detections.empty()

            dets = tracker.update(detections=dets)

            for tid, (left, top, right, bottom) in zip(dets.tracker_id, dets.xyxy):
                if tid == -1:
                    continue

                width = right - left
                height = bottom - top
                vertical = width / max(height, 1e-6) > VERTICAL_RATIO_THRESH

                if width * height > MIN_BOX_AREA and not vertical:
                    output_lines.append(
                        f"{frame_id},{int(tid)},{round(left,1)},{round(top,1)},{round(width,1)},{round(height,1)},-1,-1,-1,-1\n"
                    )

            save_path = os.path.join(save_dir, seq_name + ".txt")
        with open(save_path, "w") as f:
            f.writelines(output_lines)

    # Try to evaluate with TrackEval if GT is available
    gt_dir = os.path.join(SPORTSMOT_GT_ROOT, split)
    HOTA = IDF1 = MOTA = None

    try:
        result = evaluate_mot_sequences(
            gt_dir=gt_dir,
            tracker_dir=save_dir,
            metrics=["CLEAR", "HOTA", "Identity"],
        )

        MOTA = result.to_dict()["aggregate"]["CLEAR"]["MOTA"]
        HOTA = result.to_dict()["aggregate"]["HOTA"]["HOTA"]
        IDF1 = result.to_dict()["aggregate"]["Identity"]["IDF1"]

        print(
            f"split={split} | L:{lost_track_buffer}, I:{minimum_iou_threshold}, "
            f"C:{minimum_consecutive_frames}, D:{direction_consistency_weight}, "
            f"H:{high_conf_det_threshold}, T:{delta_t} -> "
            f"HOTA: {HOTA:.3f}, IDF1: {IDF1:.3f}, MOTA: {MOTA:.3f} (results in {save_dir})"
        )
    except Exception as e:
        print(
            f"split={split} | L:{lost_track_buffer}, I:{minimum_iou_threshold}, "
            f"C:{minimum_consecutive_frames}, D:{direction_consistency_weight}, "
            f"H:{high_conf_det_threshold}, T:{delta_t} -> "
            f"evaluation FAILED ({repr(e)}). Results saved in {save_dir}"
        )

    return {
        "split": split,
        "lost_track_buffer": lost_track_buffer,
        "minimum_iou_threshold": minimum_iou_threshold,
        "minimum_consecutive_frames": minimum_consecutive_frames,
        "direction_consistency_weight": direction_consistency_weight,
        "high_conf_det_threshold": high_conf_det_threshold,
        "delta_t": delta_t,
        "HOTA": HOTA,
        "IDF1": IDF1,
        "MOTA": MOTA,
        "output_dir": save_dir,
    }

In [ ]:
# Define OC-SORT hyperparameter search space 

lost_track_buffer_candidates = [10, 30, 60]
minimum_iou_threshold_candidates = [0.1, 0.3, 0.5]
minimum_consecutive_frames_candidates = [3, 5]
direction_consistency_weight_candidates = [0.0, 0.2, 0.5]
high_conf_det_threshold_candidates = [0.4, 0.6, 0.8]
delta_t_candidates = [1, 3]

all_candidate_lists = [
    lost_track_buffer_candidates,
    minimum_iou_threshold_candidates,
    minimum_consecutive_frames_candidates,
    direction_consistency_weight_candidates,
    high_conf_det_threshold_candidates,
    delta_t_candidates,
]

parameter_combinations = list(itertools.product(*all_candidate_lists))

print(f"Total OC-SORT parameter combinations: {len(parameter_combinations)}")

Total OC-SORT parameter combinations: 324


In [ ]:
def tune_ocsort_on_val_parallel(parameter_combinations):
    """Run OC-SORT over SportsMOT **val** for many hyperparameter sets in parallel.

    This only generates MOT-format result folders (one per combination),
    mirroring `trackers_ocsort_sportsmot_runs (1).ipynb`. Metrics must be
    computed separately (e.g., via the SportsMOT evaluation server or a
    dedicated evaluation notebook).
    """

    num_workers = os.cpu_count()
    print(f"Running {len(parameter_combinations)} OC-SORT combinations on val with {num_workers} workers")

    ctx = mp.get_context("fork")
    all_results = []

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as executor:
        futures = []
        for i, comb in enumerate(parameter_combinations):
            print(f"Submitting combination {i + 1}/{len(parameter_combinations)}")
            (
                lost_track_buffer,
                minimum_iou_threshold,
                minimum_consecutive_frames,
                direction_consistency_weight,
                high_conf_det_threshold,
                delta_t,
            ) = comb
            futures.append(
                executor.submit(
                    run_ocsort_sportsmot_once,
                    "val",
                    lost_track_buffer,
                    minimum_iou_threshold,
                    minimum_consecutive_frames,
                    direction_consistency_weight,
                    high_conf_det_threshold,
                    delta_t,
                )
            )

        for i, f in enumerate(futures):
            try:
                result = f.result()
                all_results.append(result)
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination finished.")
            except Exception as e:
                print(f"[{i + 1}/{len(parameter_combinations)}] val combination FAILED: {repr(e)}")

    df = pd.DataFrame(all_results)
    print("Validation hyperparameter sweep complete. Mapping stored in 'ocsort_sportsmot_val_tuning_df'.")
    print(df.head())

    if not df.empty:
        df.to_csv("ocsort_sportsmot_val_tuning.csv", index=False)
    else:
        print("Warning: no successful validation runs. Check paths and errors printed above.")

    return df


# Generate OC-SORT outputs for all hyperparameter combinations on **val**
ocsort_sportsmot_val_tuning_df = tune_ocsort_on_val_parallel(parameter_combinations)

Running 324 OC-SORT combinations on val with 10 workers
Submitting combination 1/324
Submitting combination 2/324
Submitting combination 3/324
Submitting combination 4/324
Submitting combination 5/324
Submitting combination 6/324
Submitting combination 7/324
Submitting combination 8/324
Submitting combination 9/324
Submitting combination 10/324
Submitting combination 11/324
Submitting combination 12/324
Submitting combination 13/324
Submitting combination 14/324
Submitting combination 15/324
Submitting combination 16/324
Submitting combination 17/324
Submitting combination 18/324
Submitting combination 19/324
Submitting combination 20/324
Submitting combination 21/324
Submitting combination 22/324
Submitting combination 23/324
Submitting combination 24/324
Submitting combination 25/324
Submitting combination 26/324
Submitting combination 27/324
Submitting combination 28/324
Submitting combination 29/324
Submitting combination 30/324
Submitting combination 31/324
Submitting combination 

## Run final run with test set to send to submission server

In [ ]:
import os
import shutil
import pandas as pd

# Load validation tuning results (must include HOTA/IDF1/MOTA columns)
df = pd.read_csv("ocsort_sportsmot_val_tuning.csv")

if "HOTA" not in df.columns or df["HOTA"].isna().all():
    raise ValueError(
        "No HOTA values found in 'ocsort_sportsmot_val_tuning.csv'. "
        "Make sure GT is available under SPORTSMOT_GT_ROOT and rerun the tuning cell."
    )

# Row with best HOTA on validation
best_idx = df["HOTA"].idxmax()
best = df.loc[best_idx]

print("Best validation row by HOTA:\n", best)
print("Best HOTA:", best["HOTA"])
print("Parameters:")
print("  lost_track_buffer        :", best["lost_track_buffer"])
print("  minimum_iou_threshold    :", best["minimum_iou_threshold"])
print("  minimum_consecutive_frames:", best["minimum_consecutive_frames"])
print("  direction_consistency_weight:", best["direction_consistency_weight"])
print("  high_conf_det_threshold  :", best["high_conf_det_threshold"])
print("  delta_t                  :", best["delta_t"])

best_params = dict(
    lost_track_buffer=int(best["lost_track_buffer"]),
    minimum_iou_threshold=float(best["minimum_iou_threshold"]),
    minimum_consecutive_frames=int(best["minimum_consecutive_frames"]),
    direction_consistency_weight=float(best["direction_consistency_weight"]),
    high_conf_det_threshold=float(best["high_conf_det_threshold"]),
    delta_t=int(best["delta_t"]),
)

print("\nRunning final OC-SORT on SportsMOT **test** split with best params:")
print(best_params)

# Run on test split with best hyperparameters
test_result = run_ocsort_sportsmot_once(
    split="test",
    **best_params,
)

print("\nTest split metrics (if GT available):")

# Create a zip for submission
submission_dir = test_result["output_dir"]
submission_zip_base = os.path.basename(submission_dir)
submission_zip = 'ocsort_sportsmot_' + submission_zip_base + ".zip"

shutil.make_archive(submission_zip_base, "zip", submission_dir)
print("\nCreated submission archive:", submission_zip)
print("Archive contains MOT-format txt files ready for upload to the evaluation server.")

Best validation row by HOTA:
 split                                                                         val
lost_track_buffer                                                              60
minimum_iou_threshold                                                         0.1
minimum_consecutive_frames                                                      3
direction_consistency_weight                                                  0.2
high_conf_det_threshold                                                       0.6
delta_t                                                                         3
HOTA                                                                     0.815854
IDF1                                                                     0.822222
MOTA                                                                     0.976134
output_dir                      OCSORT_outputs_sportsmot/val_L60_I0.1__C3_D0.2...
Name: 225, dtype: object
Best HOTA: 0.8158541061349691
Parameters:
 